In [ ]:
import joblib
import random
import optuna
from nn_models   import *
from nn_data     import *
from nn_training import *
from nn_score_and_plot import *
from torch.utils.data            import DataLoader
from sklearn.preprocessing       import StandardScaler
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
DATA_PATH = '/path'
OUT_PATH = '/path'

In [ ]:
features = KEY_FEATURES + CONTROL_FEATURES
df_train = pd.read_csv(DATA_PATH + 'train.csv')
df_val   = pd.read_csv(DATA_PATH + 'val.csv')
df_val   = df_val.dropna(subset='pm25')

X_train = np.array(df_train[features])
y_train = np.array(df_train['pm25_filled'])
X_val   = np.array(df_val[features])
y_val   = np.array(df_val['pm25'])

In [ ]:
def make_objective(X_tr, y_tr, X_vl, y_vl, n_sources, n_fkey, n_fcontrol):
    """
    Returns an Optuna objective that searches over architecture AND training
    hyperparameters using spatial 5-fold CV.
    Folds are split by monitoring station (stcode) to avoid spatial leakage.
    Scalers are re-fitted inside every fold.
    """
    def objective(trial):
        h1, h2, hout = suggest_chain(trial, ["h1", "h2", "hout"])          
        hh1, hh2     = suggest_chain(trial, ["hh1", "hh2"], low=3, high=5)
        hsk          = trial.suggest_categorical("hsk",  [8, 16, 32])
        drop_rate    = trial.suggest_float("drop_rate",  0.1, 0.3, step=0.1)
        hdrop_rate   = trial.suggest_float("hdrop_rate", 0.1, 0.3, step=0.1)
        lr           = trial.suggest_float("lr",         1e-4, 1e-2, log=True)
        lambda_sim   = trial.suggest_float("lambda_sim", 1e-2, 5e-2, step=0.01)
        batch_size   = trial.suggest_categorical("batch_size", [128, 256, 512])

        # Fit scalers inside fold — no leakage
        f_sc = StandardScaler().fit(X_tr)
        t_sc = StandardScaler().fit(y_tr.reshape(-1, 1))
        
        # Tensors & loader
        tr_dataset = MLPDataset(X_tr, y_tr, f_sc, t_sc)
        vl_dataset = MLPDataset(X_vl, y_vl, f_sc, t_sc)
        loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)

        physical_zero_std = -f_sc.mean_[:n_sources] / f_sc.scale_[:n_sources]

        # Train
        m    = TwoStageMLP(
                    n_sources  = n_sources,                    
                    n_fkey     = n_fkey,
                    n_fcontrol = n_fcontrol,         
                    h1         = h1, 
                    h2         = h2, 
                    hout       = hout, 
                    drop_rate  = drop_rate,     
                    hh1        = hh1, 
                    hh2        = hh2, 
                    hdrop_rate = hdrop_rate, 
                    hsk        = hsk,           
                    threshold  = physical_zero_std,
                    **source_sum_scaler_params(f_sc, t_sc, n_sources))
        opt  = torch.optim.Adam(m.parameters(), lr=lr)

        train_losses, best_val, best_epoch, val_losses = train_TwostageMLP(
            m, loader, opt, epochs=MAX_EPOCH, lambda_sim=lambda_sim,
            val_X=vl_dataset.X, 
            val_y=vl_dataset.y, 
            patience=PATIENCE)
        
        m.eval()
        with torch.no_grad():
            train_pred_inv = t_sc.inverse_transform(m(tr_dataset.X)[0].numpy()).reshape(-1)
            val_pred_inv   = t_sc.inverse_transform(m(vl_dataset.X)[0].numpy()).reshape(-1)
        
        train_score = calculate_score(obs=y_tr, pred=train_pred_inv)
        val_score   = calculate_score(obs=y_vl, pred=val_pred_inv)
        
        trial.set_user_attr('train_score', train_score.tolist())
        trial.set_user_attr('val_score',   val_score.tolist())
        trial.set_user_attr("h1", h1)
        trial.set_user_attr("h2", h2)
        trial.set_user_attr("hout", hout)
        trial.set_user_attr("hh1", hh1)
        trial.set_user_attr("hh2", hh2)
        trial.set_user_attr('best_epoch',  best_epoch)
        trial.set_user_attr('train_losses', train_losses)
        trial.set_user_attr('val_losses',   val_losses)
        
        print(
            f"trial {trial.number}  "
            f"best_val {best_val:.4f}  "
            f"best_epoch {best_epoch}",
            flush=True)

        return float(best_val)

    return objective

def optuna_callback(study, trial):
    if trial.state != optuna.trial.TrialState.COMPLETE:
        return
    done = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    tag  = "  ← new best" if trial.value == study.best_value else ""
    best_epoch  = trial.user_attrs.get('best_epoch', '?')
    print(f"[{done:3d}/{N_TRIALS}]  trial {trial.number:3d} "f"Val MSE {trial.value:.4f}  |  best {study.best_value:.4f} "f"best_epoch {best_epoch}{tag}")

In [ ]:
seed_everything()

N_TRIALS  = 25
TIMEOUT   = None
MAX_EPOCH = 100
PATIENCE  = 10

study = optuna.create_study(
    direction  = "minimize",
    sampler    = optuna.samplers.RandomSampler(seed=88),
    study_name = "twostage_mlp")

study.optimize(
    make_objective(X_train, y_train, X_val, y_val, N_SOURCES, N_FKEY, N_FCONTROL),
    n_trials  = N_TRIALS,
    timeout   = TIMEOUT,
    callbacks = [optuna_callback])

In [ ]:
best_trial = study.best_trial

train_curves = best_trial.user_attrs['train_losses']   
val_curves   = best_trial.user_attrs['val_losses']    

fig, ax = plt.subplots(figsize=(6, 4))
fz=16
ax.plot(train_curves, label='Train')
ax.plot(val_curves,   label='Validation')
ax.axvline(best_trial.user_attrs['best_epoch']- 1,
           color='red', linestyle='--', label='Best epoch')
ax.set_title(f'Two-stage MLP', fontsize=fz)
ax.set_xlabel('Epoch', fontsize=fz)
ax.tick_params(axis='both', labelsize=fz)
ax.grid(True, linestyle='dotted')
    
ax.legend(fontsize=fz, loc='upper right')
ax.set_ylabel('Two-stage Loss', fontsize=fz)
plt.tight_layout()

In [ ]:
print(f"Best CV loss : {study.best_value:.6f}")
print(f"Best params  : {study.best_params}")
print(f"Best epoch   : {study.best_trial.user_attrs['best_epoch']}")
print(f"h1   : {study.best_trial.user_attrs['h1']}")
print(f"h2   : {study.best_trial.user_attrs['h2']}")
print(f"hout : {study.best_trial.user_attrs['hout']}")
print(f"hh1  : {study.best_trial.user_attrs['hh1']}")
print(f"hh2  : {study.best_trial.user_attrs['hh2']}")
print(f"Train score : {study.best_trial.user_attrs['train_score']}")
print(f"Val score   : {study.best_trial.user_attrs['val_score']}")

In [ ]:
# joblib.dump(study, OUT_PATH)